# Transcriptome Wide Association Study

Transcriptome-wide association analysis (TWAS) is included as a continuation of the SuSiE-TWAS workflow. The output from TWAS is used to perform variant selection used in the causal TWAS (cTWAS) analysis. 

Input:

`--gwas_meta_data`: GWAS metadata table, one row per study × chromosome.

```
study_id                      chrom  file_path                                                             sample_size
protocol_example_twas_chr22   22     input/twas/protocol_example.twas.gwas_sumstats.chr22.tsv.gz           200000
```

| Column | Description |
|--------|-------------|
| `study_id` | GWAS study name (used in output file naming) |
| `chrom` | Chromosome number (integer, no "chr" prefix) |
| `file_path` | Path to GWAS sumstats TSV.gz |
| `sample_size` | GWAS sample size (used to scale statistics) |

`--xqtl_meta_data`: xQTL TWAS weights metadata, one row per gene.

```
#chr   start      end        region_id           TSS       original_data                                                  contexts
chr22  10000000   19000000   ENSG00000130538   15528191   input/twas/protocol_example.twas.univariate_twas_weights.rds    bulk_rnaseq
```

| Column | Description |
|--------|-------------|
| `#chr / start / end` | Analysis region (LD block coordinates) |
| `region_id` | Gene Ensembl ID |
| `TSS` | Transcription start site coordinate |
| `original_data` | Path to `*.univariate_twas_weights.rds` (from `mnm_regression.ipynb susie_twas`) |
| `contexts` | Context names covered by this file (comma-separated if multiple) |

`--ld_meta_data`: LD reference index file (same file used by `rss_analysis.ipynb`, produced by `rss_ld_sketch.ipynb`).

`--regions`: LD block BED file, one block per row.

```
chr    start      stop
chr22  10000000   19000000
```

`--xqtl_type_table`: mapping from context to xQTL type.

```
context      type
bulk_rnaseq  eQTL
```

`--ld_reference_sample_size`: sample size of the LD reference panel (used to scale the LD matrix).

`--rsq_cutoff` / `--rsq_pval_cutoff`: a gene is imputable (`is_imputable = TRUE`) if at least one method has `rsq_cv ≥ rsq_cutoff` AND `pval_cv < rsq_pval_cutoff`.

`--region-name`: run only the specified LD block (optional; omit to run all blocks in `--regions`).

## Overview

1. Run TWAS
2. Run cTWAS

## Steps

### i. Run TWAS

```bash
sos run pipeline/twas_ctwas.ipynb twas \
    --cwd output --name protocol_example \
    --gwas_meta_data input/twas/protocol_example.twas.gwas_meta.tsv \
    --xqtl_meta_data input/twas/protocol_example.twas.xqtl_meta.tsv \
    --ld_meta_data input/ld_reference/protocol_example.ld_meta_file.tsv \
    --ld_reference_sample_size 17000 \
    --regions input/twas/protocol_example.twas.LD_blocks.chr22.bed \
    --xqtl_type_table input/twas/protocol_example.twas.data_type_table.txt \
    --rsq_pval_cutoff 0.05 --rsq_cutoff 0.01 \
    --region-name chr22_10000000_19000000
```

### ii. Run cTWAS

```
sos run pipeline/twas_ctwas.ipynb ctwas \
   --cwd output/twas --name test \
   --gwas_meta_data data/twas/gwas_meta_test.tsv \
   --ld_meta_data data/ld_meta_file_with_bim.tsv \
   --xqtl_meta_data data/twas/mwe_twas_pipeline_test_small.tsv \
   --twas_weight_cutoff 0 \
   --chrom 11 \
   --regions data/twas/EUR_LD_blocks.bed \
   --region-name chr10_80126158_82231647 chr11_84267999_86714492
```

## Anticipated Results

i. Run TWAS

**`{name}.{region_id}.{gene}.twas.rds`** — `GRanges` S4 object, one file per gene.

```r
suppressPackageStartupMessages({ library(pecotmr); library(GenomicRanges) })

# one file per gene — region_id = LD block, e.g. chr22_10000000_19000000
res <- readRDS("output/twas/protocol_example.chr22_10000000_19000000.ENSG00000130538.twas.rds")
class(res)          # "GRanges"

names(mcols(res))
# [1] "qtlStudy"    "context"     "trait"       "method"
# [5] "gwasStudy"   "twasZ"       "twasPval"    "waldRatio"
# [9] "waldRatioSe" "mrPval"      "nIV"         "Q"
# [13] "I2"          "nCs"
```

| Column | Description |
|--------|-------------|
| `seqnames`, `start`, `end` | Gene genomic coordinates (LD block range) |
| `qtlStudy` | xQTL study name (from `--name`) |
| `context` | Cell type / tissue context |
| `trait` | Gene Ensembl ID |
| `method` | TWAS weight method (`mrash`, `susie`, `ensemble`, etc.) |
| `gwasStudy` | GWAS study name |
| `twasZ` | **TWAS Z-score**: **w**ᵀ **z**_GWAS / √(**w**ᵀ R **w**) |
| `twasPval` | Two-sided p-value |
| `waldRatio` / `waldRatioSe` | MR Wald ratio effect and SE (single instrument) |
| `mrPval` | MR p-value |
| `nIV` | Number of MR instrument variables |
| `Q` / `I2` | MR heterogeneity statistics |
| `nCs` | Number of credible sets contributing to the TWAS signal |

**Example result (toy data, chr22, n=49):**

```r
df <- as.data.frame(res)
df[!is.na(df$seqnames), c("context", "trait", "method", "gwasStudy", "twasZ", "twasPval")]
#       context           trait method                   gwasStudy    twasZ  twasPval
# bulk_rnaseq ENSG00000130538  mrash protocol_example_twas_chr22 1.867225 0.0618702
```

> `twasPval = 0.062` is sub-threshold — toy data (n=49) has insufficient power.
> Genome-wide significance threshold: p < 5×10⁻⁶ (Bonferroni over ~20,000 genes).

ii. Run cTWAS

`ctwas_region_chr1_z_snp_map.rds`
* includes:

1. z_snp - snp z-values per study
2. snp_map


`ctwas_region_chr1_LD_map.rds`
* includes:

1. region_id
2. LD_file
3. SNP_file


`ctwas_region_chr1_ctwas_weights.rds`
* includes ctwas weights for each gene in each study

`ctwas_region_chr1_[study_name].ctwas_boundary_genes.thin0.1.rds`
* one of these files is generated for each study. Includes:
1. chrom
2. id
3. p0
4. p1
5. molecular_id
6. weight_name
7. region_start
8. region_stop
9. region_id
10. n_regions

`ctwas_region_chr1_[study_name].ctwas_region_data.thin0.1.rds`
* one of these files is generated for each study. Include these values for each region:
1. region_id
2. chrom
3. start
4. stop
5. minpos
6. maxpos
7. thin
8. gid
9. sid
10. z_gene
11. z_snp
12. types
13. contexts
14. groups